# Phân Tích Chuỗi Thời Gian Chuyên Sâu (Advanced Time Series EDA)
## Giai đoạn 2: Đặc trưng động lực và Kiểm định tính dừng
---
Notebook này tập trung vào việc phân tích các đặc trưng động học của kinh tế Việt Nam, bao gồm tốc độ tăng trưởng, độ biến động, tính dừng và các mối tương quan trễ.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)
print("Libraries loaded.")

In [ ]:
df_gdp_sectors = pd.read_csv('../data/processed/gdp_sectors_processed.csv', index_col=0)
df_tfp = pd.read_csv('../data/processed/tfp_contribution_processed.csv', index_col=0)

# Tính toán tốc độ tăng trưởng (Growth Rate)
df_growth = df_gdp_sectors[['Giá trị: Tổng số']].pct_change() * 100
df_growth.columns = ['GDP_Growth_Rate']

print("Dữ liệu tăng trưởng đã sẵn sàng!")

### 1. Phân tích Tốc độ Tăng trưởng & Độ biến động (Volatility)
Chúng ta xem xét GDP không chỉ ở giá trị tuyệt đối mà ở tốc độ thay đổi hàng năm.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Biểu đồ tăng trưởng
ax1.plot(df_growth.index, df_growth['GDP_Growth_Rate'], marker='o', color='crimson', linewidth=2)
ax1.axhline(df_growth['GDP_Growth_Rate'].mean(), color='blue', linestyle='--', label=f"Trung bình: {df_growth['GDP_Growth_Rate'].mean():.2f}%")
ax1.set_title('Tốc độ Tăng trưởng GDP Việt Nam (%)')
ax1.legend()

# Phân phối của tăng trưởng
sns.histplot(df_growth['GDP_Growth_Rate'].dropna(), kde=True, ax=ax2, color='teal')
ax2.set_title('Phân phối Tần suất Tăng trưởng GDP')

plt.tight_layout()
plt.show()

### 2. Kiểm định Tính dừng (Stationarity)
Hầu hết các mô hình chuỗi thời gian (ARIMA) yêu cầu dữ liệu phải dừng. Chúng ta sử dụng kiểm định Augmented Dickey-Fuller (ADF).

In [ ]:
def check_stationarity(timeseries):
    print('Kết quả kiểm định ADF:')
    dftest = adfuller(timeseries.dropna(), autolag='AIC')
    dfoutput = pd.Series(dftest[0:4], index=['Test Statistic','p-value','#Lags Used','Number of Observations Used'])
    for key,value in dftest[4].items():
        dfoutput['Critical Value (%s)'%key] = value
    print(dfoutput)
    if dftest[1] <= 0.05:
        print("==> Chuỗi dữ liệu có tính dừng (Stationary)")
    else:
        print("==> Chuỗi dữ liệu KHÔNG có tính dừng (Non-Stationary)")

print("--- Kiểm định cho GDP tuyệt đối ---")
check_stationarity(df_gdp_sectors['Giá trị: Tổng số'])

print("\n--- Kiểm định cho Tốc độ tăng trưởng GDP ---")
check_stationarity(df_growth['GDP_Growth_Rate'])

### 3. Phân tích Tự tương quan (ACF & PACF)
Để xác định các tham số p, d, q cho mô hình dự báo sau này.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
plot_acf(df_growth['GDP_Growth_Rate'].dropna(), lags=15, ax=ax1)
plot_pacf(df_growth['GDP_Growth_Rate'].dropna(), lags=15, ax=ax2)
plt.show()

### 4. Tương quan liên ngành & TFP
Phân tích xem các cú sốc trong TFP có ảnh hưởng thế nào đến GDP sau các khoảng trễ (lags).

In [ ]:
# Kết hợp GDP growth và TFP growth
df_merged = pd.merge(df_growth, df_tfp[['TFP_pct']], left_index=True, right_index=True)

plt.figure(figsize=(10, 6))
sns.regplot(x='TFP_pct', y='GDP_Growth_Rate', data=df_merged, color='indigo')
plt.title('Mối tương quan giữa Đóng góp TFP và Tăng trưởng GDP')
plt.xlabel('TFP Contribution (%)')
plt.ylabel('GDP Growth Rate (%)')
plt.show()

print("Hệ số tương quan Pearson:", df_merged.corr().iloc[0,1])